In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show
from interpret.perf import ROC
from interpret.glassbox import LogisticRegression

# 1.Databetes

## 1-1 載入資料

In [48]:
df = pd.read_csv("../data/diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 1-2 分割特徵與標籤

In [49]:
y_d = df["Outcome"]
X_d = df.drop(columns=["Outcome"])

## 1-3 訓練/測試分割

In [50]:
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_d, y_d, test_size=0.2, random_state=42
)

## 1-4 資料探索

In [51]:
from interpret.data import ClassHistogram

hist_diabetes = ClassHistogram().explain_data(X_train_d, y_train_d, name='daibetes')
show(hist_diabetes)

<!-- http://127.0.0.1:7001/6268012704/ -->

## 1-4 訓練 EBM

In [52]:
from interpret.glassbox import ExplainableBoostingClassifier

ebm = ExplainableBoostingClassifier(random_state=42)
ebm.fit(X_train_d, y_train_d)

,feature_names,None
,feature_types,None
,max_bins,1024
,max_interaction_bins,64
,interactions,'3x'
,exclude,None
,validation_size,0.15
,outer_bags,14
,inner_bags,0
,learning_rate,0.015
,greedy_ratio,10.0


## 1-5 效能評估/計算AUC

In [53]:
ebm_perf = ROC(ebm).explain_perf(X_test_d, y_test_d, name='EBM')
show(ebm_perf)

<!-- http://127.0.0.1:7001/6429497056/ -->

In [54]:
y_prob_d = ebm.predict_proba(X_test_d)[:, 1]
auc = roc_auc_score(y_test_d, y_prob_d)
print(f"EBM Test AUC:{auc:.3f}")

EBM Test AUC:0.812


## 1-6 全域解釋

In [55]:
ebm_global = ebm.explain_global()
show(ebm_global)

<!-- http://127.0.0.1:7001/6422444624/ -->

## 1-7 局部解釋

In [56]:
ebm_local = ebm.explain_local(X_test_d[:5], y_test_d[:5])
show(ebm_local, 0)

<!-- http://127.0.0.1:7001/6429019472/ -->

# 2.Heart

## 2-1 載入資料+前處理

In [57]:
import pandas as pd
from interpret.glassbox import ExplainableBoostingClassifier

column_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
                'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
df = pd.read_csv('../data/heart.csv', names=column_names, na_values='?')
df = df.dropna()
df['target'] = df['target'].apply(lambda x_h: 1 if x_h > 0 else 0)

## 2-2 分割特徵與標籤

In [58]:
y_h = df["target"]
X_h = df.drop(columns=["target"])

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 2-3 訓練/測試分割

In [59]:
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

## 2-4 資料探索

In [60]:
from interpret.data import ClassHistogram

hist_heart = ClassHistogram().explain_data(X_train_h, y_train_h, name='heart')
show(hist_heart)

<!-- http://127.0.0.1:7001/6405996624/ -->

## 2-5 效能評估/計算AUC

In [61]:
ebm.fit(X_train_h, y_train_h)

,feature_names,None
,feature_types,None
,max_bins,1024
,max_interaction_bins,64
,interactions,'3x'
,exclude,None
,validation_size,0.15
,outer_bags,14
,inner_bags,0
,learning_rate,0.015
,greedy_ratio,10.0


In [62]:
ebm_perf = ROC(ebm).explain_perf(X_test_h, y_test_h)
show(ebm_perf)

<!-- http://127.0.0.1:7001/6428505680/ -->

In [63]:
y_prob_h = ebm.predict_proba(X_test_h)[:, 1]
auc = roc_auc_score(y_test_h, y_prob_h)
print(f"EBM Test AUC:{auc:.3f}")

EBM Test AUC:0.940


## 2-6 全域解釋

In [64]:
ebm_global = ebm.explain_global()
show(ebm_global)

<!-- http://127.0.0.1:7001/6424335120/ -->

## 2-7 局部解釋

In [65]:
ebm_local = ebm.explain_local(X_test_h[:5], y_test_h[:5])
show(ebm_local, 0)

<!-- http://127.0.0.1:7001/6424335360/ -->

# 3.Stroke

## 3-1 載入資料

In [66]:
df = pd.read_csv("../data/stroke.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [67]:
df = df.drop(columns=["id"]).dropna()
df.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
5,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1


## 3-2 分割特徵與標籤

In [68]:
y_s = df["stroke"]
X_s = df.drop(columns=["stroke"])
X_s = pd.get_dummies(X_s).astype(float)

## 3-3 訓練／測試分割

In [69]:
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_s, y_s, test_size=0.2, random_state=42
)

## 3-4 資料探索

In [70]:
from interpret.data import ClassHistogram

hist_stroke = ClassHistogram().explain_data(X_train_s, y_train_s, name='stroke')
show(hist_stroke)

<!-- http://127.0.0.1:7001/6424491104/ -->

## 3-5 效能評估/計算AUC

In [71]:
ebm.fit(X_train_s, y_train_s)

,feature_names,None
,feature_types,None
,max_bins,1024
,max_interaction_bins,64
,interactions,'3x'
,exclude,None
,validation_size,0.15
,outer_bags,14
,inner_bags,0
,learning_rate,0.015
,greedy_ratio,10.0


In [72]:
ebm_perf = ROC(ebm).explain_perf(X_test_s, y_test_s)
show(ebm_perf)

<!-- http://127.0.0.1:7001/6429279376/ -->

In [73]:
y_prob_s = ebm.predict_proba(X_test_s)[:, 1]
auc = roc_auc_score(y_test_s, y_prob_s)
print(f"EBM Test AUC:{auc:.3f}")

EBM Test AUC:0.842


## 3-6 全域解釋

In [74]:
ebm_global = ebm.explain_global()
show(ebm_global)

<!-- http://127.0.0.1:7001/6434327248/ -->

## 3-7 局部解釋

In [75]:
ebm_local = ebm.explain_local(X_test_s[:5], y_test_s[:5])
show(ebm_local, 0)

<!-- http://127.0.0.1:7001/6434328816/ -->

# 比較Dashboard

In [76]:
show([hist_heart, hist_stroke, hist_diabetes])

<!-- http://127.0.0.1:7001/6404852288/ -->
 Open in new window

# AUC對比表

In [77]:
from sklearn.metrics import roc_auc_score
import pandas as pd

def get_auc(model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, y_prob)

In [78]:
from interpret.glassbox import LogisticRegression, ClassificationTree

lr_h = LogisticRegression(random_state=42, penalty='l1', solver='liblinear')
lr_h.fit(X_train_h, y_train_h)

lr_s = LogisticRegression(random_state=42, penalty='l1', solver='liblinear')
lr_s.fit(X_train_s, y_train_s)

lr_d = LogisticRegression(random_state=42, penalty='l1', solver='liblinear')
lr_d.fit(X_train_d, y_train_d)

In [79]:
from sklearn.metrics import roc_auc_score
import pandas as pd

def auc(model, X_test, y_test):
    return roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

auc_df = pd.DataFrame([
    ["Heart", auc(lr_h, X_test_h, y_test_h)],
    ["Stroke", auc(lr_s, X_test_s, y_test_s)],
    ["Diabetes", auc(lr_d, X_test_d, y_test_d)]
], columns=["Model", "AUC"])

auc_df

,Model,AUC
0,Heart,0.936343
1,Stroke,0.853728
2,Diabetes,0.813958
